# Closed-unit extension: all 15 networks and training tests

This review keeps the frozen seven-input forest unchanged. It tests 15 segmented single-centerline representations and six adjacent closed-unit pairs. New cases add collapse configurations, **not independent networks**.

## What goes into the predictors?

Your proposed exclusion is implemented: unchanged **inter-block connectors and upstream/downstream tails are excluded from the new composite predictors**, while remaining in RAPID routing. Interior reaches belonging to selected simple/full-compound blocks contribute to the predictors.

For route probabilities, retain the connected graph. Give excluded edges zero *length reward* when calculating travelled-length moments; do not pretend their physical length is zero in RAPID. Width moments use only selected edges and their physical lengths. Entropy and dominance use selected complete splits. This retains valid probability propagation through confluences without counting connector geometry.

Existing 2,170 comparisons retain their established outer-domain predictors. In particular, we have not removed unchanged branches inside the existing containing-compound domains of compound-member comparisons. That would be a separate change to the baseline.

## Why one common window?

Each network has one baseline and common integration boundaries for **all** its states. If a newly added fast response starts before the old window, using the old interval would truncate part of that response. Giving just that state a longer interval would make volume/centroid comparisons inconsistent.

We therefore extend the shared window where needed and recompute the response metrics for the original states as well. The primary threshold remains 0.5%; 1% and 2% are retained separately. Boundaries use departures from initial/final equilibrium with the established sustained-onset rule, not individual peak matching. Peak timing always compares global maxima.

In these runs, eight networks need an earlier start (1–94 minutes); none needs a later end or different baseline. Among the original 2,170 comparisons, centroid shifts change by only 0.075 seconds on average, at most 10.25 seconds. Peak timing and peak-excess changes are unchanged. Example 54 retains its finite-window and crest-switch interpretation.


In [2]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'RAPID').is_dir())
OUT = ROOT / 'RAPID/outputs/hydrograph_closed_unit_extension_20261019'
responses = pd.read_parquet(OUT / 'responses.parquet')
metrics = pd.read_parquet(OUT / 'state_metrics.parquet')
scores = pd.read_csv(OUT / 'augmentation/scores.csv')
windows = pd.read_csv(OUT / 'window_audit.csv', dtype={'example': str})
display(windows.query("window == '0.5%'")[['example','earlier_start_minutes','later_end_minutes','old_baseline_cms','new_baseline_cms']])

,example,earlier_start_minutes,later_end_minutes,old_baseline_cms,new_baseline_cms
0,02,17.0,0.0,946.986290,946.986290
3,03,24.0,0.0,1520.700000,1520.700000
6,05,94.0,0.0,328.700000,328.700000
9,13,0.0,0.0,647.900000,647.900000
12,15,0.0,0.0,1050.000000,1050.000000
15,21,52.0,0.0,8135.052740,8135.052740
18,22,82.0,0.0,18825.528487,18825.528487
21,37,0.0,0.0,1469.833117,1469.833117
24,41,0.0,0.0,4729.857276,4729.857276
27,45,0.0,0.0,1066.031786,1066.031786


## All-network routing results

All 63 routed states completed and settled: 15 parents, 15 legacy uniform centerlines, 15 segmented centerlines, six pairs and 12 individual controls. Reference reruns reproduce the original hydrographs. Every segmented version still advances the centroid relative to the parent, but less than the legacy uniform representation.

Twenty-seven saved block geometries in six networks had small endpoint offsets (maximum 14.06 m). Only the new variants restore endpoints to their explicitly mapped parent nodes; interior vertices, boundary heads and calibration are retained. Length/slope/routing parameters are recalculated. Maximum section-length adjustment is 0.829%. Unchanged reaches and original files remain untouched. See `anchor_restoration.csv` and `reference_reproduction.csv` for the audits.


In [4]:
r = responses.query("window == '0.5%' and cohort in ['legacy_uniform','segmented']")
display(r.pivot(index='example', columns='cohort', values='centroid_shift_hours').round(3))
px.bar(r, x='example', y='centroid_shift_hours', color='cohort', barmode='group',
       labels={'centroid_shift_hours':'Centroid shift (hours; child − parent)'},
       title='Uniform versus segmented single centerlines').show()

cohort,legacy_uniform,segmented
example,,
02,-2.784,-0.966
03,-2.897,-2.179
05,-8.408,-6.535
13,-11.347,-6.980
15,-6.557,-4.460
21,-4.689,-1.728
22,-11.583,-5.263
37,-8.008,-7.764
41,-17.518,-13.244


In [5]:
EXAMPLE = '22'  # Any of the 15 networks, e.g. '03', '22', '46'
m = metrics[(metrics.example == EXAMPLE) & (metrics.window == '0.5%')]
fig = go.Figure()
for state, label in [('S000_base','Parent'), ('fully_collapsed','Legacy uniform'), ('all_closed_units_collapsed','Segmented')]:
    row = m[m.state == state].iloc[0]
    h = pd.read_parquet(row.outlet)
    t = pd.to_datetime(h.interval_start_utc) + (pd.to_datetime(h.interval_end_utc)-pd.to_datetime(h.interval_start_utc))/2
    fig.add_scatter(x=t, y=h.q_outlet_cms, name=label, mode='lines')
start, end = pd.Timestamp(m.iloc[0].rise_utc), pd.Timestamp(m.iloc[0].fall_utc)
fig.add_vrect(x0=start, x1=end, fillcolor='grey', opacity=.08, line_width=0)
fig.update_layout(title=f'{EXAMPLE}: same forcing, no peak alignment; shaded common window',
                  xaxis_title='Time (UTC)', yaxis_title='Discharge (m³/s)')
fig.update_xaxes(range=[start, end])
fig.show()

## Does adding cases improve predictions?

Four training sets use exactly the frozen forest settings. Each fold holds out **all states of one network**, including its added pairs/centerline. Preprocessing and target standardization use training data only. Networks retain equal total training and evaluation weight. Scores are reported separately for original cases, pairs and segmented centerlines; pooling them would allow large new responses to dominate R².

The training target is full excess-volume centroid shift. Peak timing is a transfer evaluation of those same predictions, not a separately trained peak model. Relative error uses a one-minute denominator floor; it is not ordinary unbuffered MAPE. MAE/RMSE below are minutes. Settings were previously selected on the original data: this is a fixed-setting augmentation experiment, not fresh nested model selection.


In [7]:
display(scores.query("response == 'centroid_shift_hours' and subset == 'all'")
        [['variant','cohort','n','networks','r2','mae','relative_error_pct']].round(3))
# Other responses and exclusion sensitivities remain available:
# display(scores.query("response == 'peak_shift_hours' and subset == 'all'"))
network_scores = pd.read_csv(OUT / 'augmentation/network_scores.csv', dtype={'example':str})
ns = network_scores.query("response == 'centroid_shift_hours' and cohort == 'original'")
display(ns.pivot(index='example', columns='variant', values='mae_minutes').round(3))

,variant,cohort,n,networks,r2,mae,relative_error_pct
0,original_only,original,2170,15,0.615,2.823,63.741
3,original_only,pair,6,3,0.881,11.300,45.814
6,original_only,segmented,15,15,-0.818,342.740,65.198
18,plus_both,original,2170,15,-25.486,16.313,1235.639
21,plus_both,pair,6,3,-12.084,65.290,93.760
24,plus_both,segmented,15,15,0.186,226.542,51.317
36,plus_pairs,original,2170,15,0.619,2.820,63.904
39,plus_pairs,pair,6,3,0.870,11.643,45.941
42,plus_pairs,segmented,15,15,-0.792,339.236,64.435
54,plus_segmented,original,2170,15,-25.305,16.259,1228.771


variant,original_only,plus_both,plus_pairs,plus_segmented
example,,,,
02,1.271,1.624,1.312,1.632
03,1.430,1.596,1.430,1.603
05,4.935,5.163,4.892,5.166
13,2.399,2.394,2.360,2.467
15,1.914,1.935,1.889,1.945
21,3.347,2.998,3.437,2.949
22,1.726,202.395,1.736,201.336
37,2.866,2.804,2.857,2.810
41,3.484,3.475,3.490,3.513


## Interpretation and next decision

- **Six pairs:** original-case R² changes from 0.6145 to 0.6189, MAE from 2.823 to 2.820 minutes, and floored relative error from 63.74% to 63.90%. This is essentially unchanged performance, not convincing improvement. Only three networks supply pairs. The original-only model already predicts these six held-network pairs reasonably (R² 0.881), though relative error is 45.8%.
- **Fifteen segmented centerlines:** their own held-network R² improves from −0.818 to 0.222, but original-case performance deteriorates severely. Most of the damage is in network 22: its original-case MAE rises from 1.73 to 201.34 minutes. For example, `S262_unit_287` has a −0.573-hour centroid shift but is predicted at −6.675 hours after augmentation. This is not evidence that the new routed hydrographs are invalid.
- Original large outer-domain cases in 22 retain substantial child entropy and almost unchanged length ratios; the new segmented cases eliminate route entropy and route-length SD. A forest nevertheless generalizes poorly between these populations under the fixed configuration. This needs investigation, rather than promoting the augmented model.
- Adding the large responses also increases the training target standard deviation from about 0.19 to 0.76 hours. Because the existing model standardizes its target and applies `ccp_alpha=0.001`, this changes the effective pruning scale in physical hours. Domain scale, training weights and pruning are candidate explanations; these runs do not isolate their contributions.

**Keep the current frozen model.** The next useful extension is more adjacent closed-unit pairs across all 15 networks, spanning edited sizes and collapse strengths. Treat the segmented whole-network cases as a separate stress-test cohort for now. Before mixing them into training again, isolate target-scaling/pruning effects and diagnose the network-22 failures. Connector-free geometry remains the agreed definition for these new composite predictors; connector routing remains physically present.

Validation: five focused geometry/predictor tests passed; all 15 single-path topology and routing preparations passed; joins are one-to-one with no missing targets among the 2,170 baseline and 21 new predictor rows. The two historically excluded predictor records remain excluded. No frozen artifacts or historical response tables were overwritten.

Reproduction scripts: `prepare_all.py`, `responses.py`, `augmentation.py` in `RAPID/investigations/closed_unit_extension_20261019/`. Routing command and source provenance are saved in the output manifests and plan. The augmentation script resumes fingerprinted checkpoints.


## Follow-up: why did R² become −25?

Targeted held-network-22 controls identify a strong domain-size shortcut. Network 22 supplies 98.74% of weighted squared error after segmented augmentation; 341 of its 408 original cases have errors above one hour. Excluding 22 from evaluation, original-only R² is 0.6163 and augmented R² is 0.6538 (evaluation exclusion only, not retraining without 22).

In the augmented held-22 forest, 417/500 trees first split on parent-domain size; its impurity importance is 72.7% (diagnostic training importance, not SHAP or causal importance). Existing cases in 22 can share a 102.57 km outer parent domain while changing only a small part of it. New segmented cases have large edited domains and collapse all selected blocks. Large size therefore becomes misleading when generalized to these held-out partial edits.

Controlled fits on the same held-22 split:

| Training / control | Original-case MAE in 22 (minutes) |
|---|---:|
| Original training | 1.726 |
| Add segmented centerlines | 201.336 |
| Add segmented; preserve physical pruning penalty | 201.182 |
| Add segmented; no pruning | 201.310 |
| Add segmented; remove domain size | 2.038 |

Thus target scaling/pruning does not explain the catastrophe. Removing size nearly eliminates it in this targeted test, supporting the size-shortcut explanation. This does **not** establish that size should be removed from the final model: no full cross-network ablation has been run, and size was useful in the baseline. More consistent accounting of edited extent versus containing-domain size is a candidate next investigation. Keep the frozen model unchanged.

Scripts: `diagnose_augmentation.py`, `diagnose_size_shortcut.py`. Detailed tables: `augmentation/diagnostic22/` in the extension outputs. These controls were chosen after inspecting network-22 errors and are exploratory diagnostics, not independent model-selection evidence.


## Three size choices tested across all networks

See [the executed size-definition comparison](hydrograph_size_definition_test.ipynb). Current size, no size and edited-footprint size were compared with the other six inputs fixed. Edited size largely fixes the network-22 catastrophe, but augmented training still underperforms the frozen baseline. Removing size performs poorly across other networks. Keep the frozen model; paired-unit expansion and whole-network stress testing remain separate.

## Segmented-only retuning completed

See [segmented-only nested forest search](hydrograph_segmented_model_search.ipynb). Paired units were excluded. All three size options were tested at fixed settings, followed by 24-candidate grouped nested tuning across all 15 networks. Edited size was selected in every outer fold, but retuning did not improve on fixed edited-size performance. The segmented cases remain an active investigation; the frozen model is unchanged.